###Task 2.1: Build & Load Database

In [23]:
import sqlite3

conn=sqlite3.connect('noshowguard.db')
cursor=conn.cursor()
query="""
CREATE TABLE clinics(clinic_id VARCHAR(10) PRIMARY KEY,
    clinic_name VARCHAR(100),
    city VARCHAR(100));
"""
cursor.execute(query)
conn.commit()
print("Database connection successful")

Database connection successful


In [24]:
query = """
CREATE TABLE appointments (
    appointment_id VARCHAR(10) PRIMARY KEY,
    patient_id VARCHAR(10),
    clinic_id VARCHAR(10),
    booked_on DATE,
    appt_date DATE,
    age INTEGER,
    gender VARCHAR(10),
    distance_km REAL,
    fee_inr REAL,
    booking_channel VARCHAR(30),
    visit_type VARCHAR(40),
    reminder_sent VARCHAR(10),
    prior_no_shows INTEGER,
    status VARCHAR(20),
    lead_days INTEGER,
    no_show INTEGER,
    FOREIGN KEY (clinic_id)
    REFERENCES clinics(clinic_id)
);
"""

cursor = conn.cursor()
cursor.execute(query)
conn.commit()


In [25]:
with open('load_data.sql','r')as file:
    sql_script=file.read()
conn.executescript(sql_script)

print("Clinics loaded:", cursor.execute(
    "SELECT COUNT(*) FROM clinics"
).fetchone()[0])
print("Appointments loaded:", cursor.execute(
    "SELECT COUNT(*) FROM appointments"
).fetchone()[0])



Clinics loaded: 8
Appointments loaded: 956


In [26]:
try:
    cursor.execute("""
    INSERT INTO appointments
    SELECT * FROM appointments
    WHERE appointment_id = 'A0001'
    """)
    conn.commit()
except sqlite3.IntegrityError as e:
    print("Duplicate key error:", e)

Duplicate key error: UNIQUE constraint failed: appointments.appointment_id


###Task 2.2 — No-show rate by clinic

In [27]:
query_2_2 = """
SELECT
    c.clinic_id,
    c.clinic_name,
    COUNT(*) AS appointments,
    SUM(a.no_show) AS no_shows,
    ROUND(100.0 * SUM(a.no_show) / COUNT(*), 3) AS no_show_pct
FROM appointments a
JOIN clinics c
    ON a.clinic_id = c.clinic_id
GROUP BY c.clinic_id, c.clinic_name
HAVING COUNT(*) >= 95
ORDER BY no_show_pct DESC
"""

result_2_2 = cursor.execute(query_2_2).fetchall()

for row in result_2_2:
    print(row)

('C02', None, 143, 39, 27.273)
('C04', None, 128, 31, 24.219)
('C01', None, 160, 36, 22.5)
('C07', None, 98, 19, 19.388)
('C06', None, 139, 21, 15.108)
('C05', None, 128, 18, 14.063)


###Task 2.3 — Booking lead-time analysis


In [28]:
query_2_3 = """
SELECT
    CASE
        WHEN lead_days BETWEEN 1 AND 7 THEN '1-7'
        WHEN lead_days BETWEEN 8 AND 14 THEN '8-14'
        WHEN lead_days BETWEEN 15 AND 30 THEN '15-30'
        WHEN lead_days >= 31 THEN '31+'
    END AS lead_time_band,
    COUNT(*) AS total_appointments,
    SUM(no_show) AS total_no_shows,
    ROUND(100.0 * SUM(no_show) / COUNT(*), 3) AS no_show_pct
FROM appointments
WHERE lead_days >= 1
GROUP BY lead_time_band
ORDER BY MIN(lead_days)
"""

result_2_3 = cursor.execute(query_2_3).fetchall()

for row in result_2_3:
    print(row)

('1-7', 378, 58, 15.344)
('8-14', 293, 49, 16.724)
('15-30', 246, 72, 29.268)
('31+', 39, 23, 58.974)


###Task 2.4 — Month-by-month trend analysis

In [29]:
query_2_4 = """
WITH monthly AS (
    SELECT
        SUBSTR(appt_date, 1, 7) AS month,
        COUNT(*) AS total_appointments,
        SUM(no_show) AS total_no_shows,
        ROUND(100.0 * SUM(no_show) / COUNT(*), 3) AS no_show_pct
    FROM appointments
    GROUP BY SUBSTR(appt_date, 1, 7)
),
monthly_changes AS (
    SELECT
        month,
        total_appointments,
        total_no_shows,
        no_show_pct,
        LAG(no_show_pct) OVER (ORDER BY month) AS previous_month_pct
    FROM monthly
)
SELECT
    month,
    total_appointments,
    total_no_shows,
    no_show_pct,
    ROUND(no_show_pct - previous_month_pct, 3) AS change_percentage_points
FROM monthly_changes
ORDER BY month
"""

result_2_4 = cursor.execute(query_2_4).fetchall()

for row in result_2_4:
    print(row)

('2025-01', 152, 35, 23.026, None)
('2025-02', 146, 28, 19.178, -3.848)
('2025-03', 140, 27, 19.286, 0.108)
('2025-04', 169, 35, 20.71, 1.424)
('2025-05', 190, 43, 22.632, 1.922)
('2025-06', 159, 34, 21.384, -1.248)


###Task 2.5 — Intra-city clinic ranking


In [30]:
query_2_5 = """
WITH clinic_rates AS (
    SELECT
        c.clinic_id,
        c.clinic_name,
        c.city,
        COUNT(*) AS total_appointments,
        SUM(a.no_show) AS total_no_shows,
        100.0 * SUM(a.no_show) / COUNT(*) AS no_show_pct
    FROM appointments a
    JOIN clinics c
        ON a.clinic_id = c.clinic_id
    WHERE c.city IS NOT NULL
    GROUP BY c.clinic_id, c.clinic_name, c.city
),
ranked_clinics AS (
    SELECT
        clinic_id,
        clinic_name,
        city,
        total_appointments,
        total_no_shows,
        ROUND(no_show_pct, 3) AS no_show_pct,
        RANK() OVER (
            PARTITION BY city
            ORDER BY no_show_pct DESC
        ) AS city_rank
    FROM clinic_rates
)
SELECT
    clinic_id,
    clinic_name,
    city,
    total_appointments,
    total_no_shows,
    no_show_pct,
    city_rank,
    CASE
        WHEN no_show_pct > (
            SELECT 100.0 * SUM(no_show) / COUNT(*)
            FROM appointments
        ) THEN 'above'
        WHEN no_show_pct < (
            SELECT 100.0 * SUM(no_show) / COUNT(*)
            FROM appointments
        ) THEN 'below'
        ELSE 'equal'
    END AS comparison_with_global
FROM ranked_clinics
ORDER BY city, city_rank
"""

result_2_5 = cursor.execute(query_2_5).fetchall()

for row in result_2_5:
    print(row)

('2025-06', 159, 34, 21.384, -1.248)


In [31]:
conn.commit()
cursor.close()
conn.close()

print("All SQL verification tasks completed!")

All SQL verification tasks completed!
